# Computational Theory: SHA-256

In [ ]:
import numpy as np

## Problem 1: Representing SHA-256 Data

### 32-bit word

SHA-256 works on 32-bit words (FIPS 180-4 Section 2.2), with addition modulo 2³² (Section 3.2). A word here is a `np.uint32` and because it has a fixed width, addition wraps on its own without a mask. `np.errstate(over="ignore")` turns off the overflow warning that NumPy gives when a scalar wraps because the wrap is required by Section 3.2, not a bug being hidden. `~` also flips exactly 32 bits, which Ch (Section 4.1.2) needs. On a plain Python int, `~0` gives `-1` instead. Plain ints that are too large for 32 bits raise an `OverflowError`, which is why they are created with `np.uint32()`.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[NumPy, Data types: Overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)

In [ ]:
# H0, the first word of the initial hash value
w = np.uint32(0x6A09E667)
# Prints the word as 8 hex digits/32 bits
print(f"{w:08x}")

# Shows that addition wraps around at 2^32
with np.errstate(over="ignore"):
    wrapped = np.uint32(0xFFFFFFFF) + np.uint32(1)
print("After wrap:", hex(wrapped))
assert wrapped == 0

# Ch (FIPS 180-4 Section 4.1.2) uses NOT; ~ on uint32 flips all 32 bits, 
# but on a plain int it goes negative
print("~ uint32:", hex(~np.uint32(0)))
print("~ int   :", ~0)

6a09e667
After wrap: 0x0
~ uint32: 0xffffffff
~ int   : -1


### Sequences of 32-bit words
SHA-256 handles words in fixed-length groups. The initial hash value H⁽⁰⁾ is 8 words (FIPS 180-4 Section 5.3.3) and the round constants K are 64 words (Section 4.2.2). The message schedule W is 64 words (Section 6.2.2), and a 512-bit message block is 16 words (Section 5.2.1). A sequence is a NumPy array with `dtype=np.uint32`, so each element is a 32-bit word.

An array suits this better than a Python list for two reasons. First, `+` means something different. On lists, `[1, 2] + [3, 4]` joins them end to end and gives `[1, 2, 3, 4]`. On arrays, `+` adds element by element, which is what the standard means when it adds two sequences of words, for example when it updates the hash value (Section 6.2.2). Array addition also wraps each word modulo 2³² without a warning, so `np.errstate` isn't needed. Second, an array's size is set when it is created. That matches the fixed sizes above, and a sequence can't grow or shrink by accident.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[NumPy, *the absolute basics for beginners*](https://numpy.org/doc/stable/user/absolute_beginners.html)

In [ ]:
# H(0), the initial hash value (Section 5.3.3), is 8 words
H = np.array([0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
              0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19],
             dtype=np.uint32)
# Format each word as 8 hex digits (32 bits), separated by spaces
print(" ".join(f"{h:08x}" for h in H))
# 8 words x 4 bytes each = 32 bytes = 256 bits, the size of a SHA-256 hash
assert H.nbytes * 8 == 256

# + joins lists but adds arrays element by element
print("list  +:", [1, 2] + [3, 4])
a = np.array([1, 2], dtype=np.uint32)
b = np.array([3, 4], dtype=np.uint32)
print("array +:", a + b)

# Array addition wraps each word at 2^32
a = np.array([0xFFFFFFFF, 1], dtype=np.uint32)
b = np.array([1, 1], dtype=np.uint32)
# 0xFFFFFFFF + 1 = 2^32, which doesn't fit in 32 bits, so it wraps to 0
# 1 + 1 = 2 is unaffected
print("a + b:", [hex(x) for x in a + b])
assert list(a + b) == [0, 2]

6a09e667 bb67ae85 3c6ef372 a54ff53a 510e527f 9b05688c 1f83d9ab 5be0cd19
list  +: [1, 2, 3, 4]
array +: [4 6]
a + b: ['0x0', '0x2']


## Problem 2: SHA-256 Bitwise Operations

## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256